In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from bin.sampling import calculate_sample_size

In [2]:
def add_source_column(df: pd.DataFrame) -> None:
    """Add source column to given dataframe."""
    df.loc[df["notebook"].str.contains(r"^data/assert"), "source"] = "GH"
    df.loc[df["notebook"].str.contains(r"^data/quaranta"), "source"] = "KG"

In [ ]:
stats = pd.read_csv(
    "data/shome2023notebook/stats.csv",
    header=None,
    names=["notebook", "num_cells"]).reset_index(drop=True)
add_source_column(stats)
stats

,notebook,num_cells,source
0,data/assert_notebooks/rohitashwachaks/MIS-382N...,20,GH
1,data/assert_notebooks/rohitashwachaks/MIS-382N...,20,GH
2,data/assert_notebooks/JunchuanYu/Deep-learning...,9,GH
3,data/assert_notebooks/JunchuanYu/Deep-learning...,8,GH
4,data/assert_notebooks/JunchuanYu/Deep-learning...,9,GH
...,...,...,...
195570,data/quaranta2021kgtorrent/KT_dataset/lbroncha...,18,KG
195571,data/quaranta2021kgtorrent/KT_dataset/naomilin...,8,KG
195572,data/quaranta2021kgtorrent/KT_dataset/octavios...,18,KG
195573,data/quaranta2021kgtorrent/KT_dataset/valcoder...,22,KG


In [3]:
asserts = pd.read_csv(
    "data/shome2023notebook/asserts.csv",
    header=None,
    names=["notebook", "stmt"]
).reset_index(drop=True)
asserts["type"] = "assert"
asserts

,notebook,stmt,type
0,data/assert_notebooks/rohitashwachaks/MIS-382N...,"assert train_loss < 0.5, train_loss",assert
1,data/assert_notebooks/rohitashwachaks/MIS-382N...,"assert train_acc <= 1 and train_acc > 0.7, tra...",assert
2,data/assert_notebooks/rohitashwachaks/MIS-382N...,"assert test_acc <= 1 and test_acc > 0.7, test_acc",assert
3,data/assert_notebooks/rohitashwachaks/MIS-382N...,"assert train_loss < 0.5, train_loss",assert
4,data/assert_notebooks/rohitashwachaks/MIS-382N...,"assert train_acc <= 1 and train_acc > 0.7, tra...",assert
...,...,...,...
89601,data/quaranta2021kgtorrent/KT_dataset/ashishpa...,assert shape[2] == 3,assert
89602,data/quaranta2021kgtorrent/KT_dataset/aryansha...,assert type(self.targetName) == str,assert
89603,data/quaranta2021kgtorrent/KT_dataset/aryansha...,assert type(self.colnames) == str,assert
89604,data/quaranta2021kgtorrent/KT_dataset/aryansha...,assert self.colnames in X.columns,assert


In [4]:
lasts = pd.read_csv(
    "data/shome2023notebook/lasts.csv",
    header=None,
    names=["notebook", "stmt"]
).reset_index(drop=True)
lasts["type"] = "last"
lasts

,notebook,stmt,type
0,data/assert_notebooks/rohitashwachaks/MIS-382N...,"(X.sum(0, keepdim=True), X.sum(1, keepdim=True))",last
1,data/assert_notebooks/rohitashwachaks/MIS-382N...,"(X_prob, X_prob.sum(1))",last
2,data/assert_notebooks/rohitashwachaks/MIS-382N...,"y_hat[[0, 1], y]",last
3,data/assert_notebooks/rohitashwachaks/MIS-382N...,"cross_entropy(y_hat, y)",last
4,data/assert_notebooks/rohitashwachaks/MIS-382N...,"accuracy(y_hat, y) / len(y)",last
...,...,...,...
1004085,data/quaranta2021kgtorrent/KT_dataset/talk2ran...,data.head(),last
1004086,data/quaranta2021kgtorrent/KT_dataset/talk2ran...,data.head(),last
1004087,data/quaranta2021kgtorrent/KT_dataset/talk2ran...,data.describe(),last
1004088,data/quaranta2021kgtorrent/KT_dataset/talk2ran...,"classifier.fit(X_train, y_train)",last


In [5]:
data = pd.concat([asserts, lasts], ignore_index=True)
add_source_column(data)
data

,notebook,stmt,type,source
0,data/assert_notebooks/rohitashwachaks/MIS-382N...,"assert train_loss < 0.5, train_loss",assert,GH
1,data/assert_notebooks/rohitashwachaks/MIS-382N...,"assert train_acc <= 1 and train_acc > 0.7, tra...",assert,GH
2,data/assert_notebooks/rohitashwachaks/MIS-382N...,"assert test_acc <= 1 and test_acc > 0.7, test_acc",assert,GH
3,data/assert_notebooks/rohitashwachaks/MIS-382N...,"assert train_loss < 0.5, train_loss",assert,GH
4,data/assert_notebooks/rohitashwachaks/MIS-382N...,"assert train_acc <= 1 and train_acc > 0.7, tra...",assert,GH
...,...,...,...,...
1093691,data/quaranta2021kgtorrent/KT_dataset/talk2ran...,data.head(),last,KG
1093692,data/quaranta2021kgtorrent/KT_dataset/talk2ran...,data.head(),last,KG
1093693,data/quaranta2021kgtorrent/KT_dataset/talk2ran...,data.describe(),last,KG
1093694,data/quaranta2021kgtorrent/KT_dataset/talk2ran...,"classifier.fit(X_train, y_train)",last,KG


In [6]:
# free up memory
del asserts, lasts

# Raw Numbers

This section contains the raw numbers useful for reporting in the paper.

## How many notebooks from GH and KG?

In [2]:
!grep '^INPUT' logs/data-collection.log| grep 'assert_notebooks' |wc -l
!grep '^INPUT' logs/data-collection.log| grep 'quaranta2021kgtorrent' |wc -l

   49090
  248761


## How many notebooks after the notebook filters?

In [19]:
stats.loc[stats["source"] == "GH"].shape

(25947, 3)

In [20]:
stats.loc[stats["source"] == "KG"].shape

(169628, 3)

## How many assert statements?

In [11]:
data.loc[data["type"] == "assert"].groupby("source")["stmt"].count()

source
GH    80034
KG     9572
Name: stmt, dtype: int64

### How many after dropping duplicates?

In [12]:
data.drop_duplicates(subset=["stmt"]).loc[data["type"] == "assert"].groupby("source")[
    "stmt"
].count()

source
GH    23837
KG     3278
Name: stmt, dtype: int64

## How many last statements?

In [13]:
data.loc[data["type"] == "last"].groupby("source")["stmt"].count()

source
GH    140951
KG    863113
Name: stmt, dtype: int64

### How many after dropping duplicates?

In [14]:
data.drop_duplicates(subset=["stmt"]).loc[data["type"] == "last"].groupby("source")[
    "stmt"
].count()

source
GH     57072
KG    316521
Name: stmt, dtype: int64

# Clustering

In [15]:
data = pd.read_csv("data/shome2023notebook/clusters-dedup.csv", index_col=0)
data

,notebook,stmt,source,CALL,CGH,CKG
0,data/assert_notebooks/rohitashwachaks/MIS-382N...,"assert train_loss < 0.5, train_loss",GH,831,39.0,NaN
1,data/assert_notebooks/rohitashwachaks/MIS-382N...,"assert train_acc <= 1 and train_acc > 0.7, tra...",GH,-1,-1.0,NaN
2,data/assert_notebooks/rohitashwachaks/MIS-382N...,"assert test_acc <= 1 and test_acc > 0.7, test_acc",GH,-1,-1.0,NaN
6,data/assert_notebooks/JunchuanYu/Deep-learning...,assert p > 0 and p < 1,GH,-1,107.0,NaN
7,data/assert_notebooks/JunchuanYu/Deep-learning...,"assert stride in [1, 2]",GH,1005,72.0,NaN
...,...,...,...,...,...,...
1093671,data/quaranta2021kgtorrent/KT_dataset/deepgoja...,df.groupby('smoker')['charges'].mean(),KG,964,NaN,879.0
1093672,data/quaranta2021kgtorrent/KT_dataset/deepgoja...,df.groupby('children')['charges'].mean(),KG,964,NaN,879.0
1093673,data/quaranta2021kgtorrent/KT_dataset/deepgoja...,"sns.countplot(data=df, x='region', hue='smoker')",KG,688,NaN,708.0
1093674,data/quaranta2021kgtorrent/KT_dataset/deepgoja...,df['children'].value_counts().plot(kind='bar'),KG,610,NaN,404.0


## How many clusters for GH and KG?

In [24]:
from bin.sampling import collapse_clusters

In [25]:
data[data["CGH"].notna()]

,notebook,stmt,source,CALL,CGH,CKG
0,data/assert_notebooks/rohitashwachaks/MIS-382N...,"assert train_loss < 0.5, train_loss",GH,831,39.0,NaN
1,data/assert_notebooks/rohitashwachaks/MIS-382N...,"assert train_acc <= 1 and train_acc > 0.7, tra...",GH,-1,-1.0,NaN
2,data/assert_notebooks/rohitashwachaks/MIS-382N...,"assert test_acc <= 1 and test_acc > 0.7, test_acc",GH,-1,-1.0,NaN
6,data/assert_notebooks/JunchuanYu/Deep-learning...,assert p > 0 and p < 1,GH,-1,107.0,NaN
7,data/assert_notebooks/JunchuanYu/Deep-learning...,"assert stride in [1, 2]",GH,1005,72.0,NaN
...,...,...,...,...,...,...
230546,data/assert_notebooks/zzzzzzzzzzzzz/numerical-...,sample_csr.sum(axis=1),GH,466,104.0,NaN
230547,data/assert_notebooks/zzzzzzzzzzzzz/skoltech-m...,current_eigen,GH,-1,-1.0,NaN
230549,data/assert_notebooks/zzzzzzzzzzzzz/skoltech-m...,(mu - current_eigen).norm(),GH,637,174.0,NaN
230552,data/assert_notebooks/zzzzzzzzzzzzz/skoltech-m...,target_points,GH,-1,247.0,NaN


In [26]:
collapse_clusters(data=data[data["CGH"].notna()], col="CGH")

Large clusters  : 50
Coverage        : 69.8%


In [27]:
collapse_clusters(data=data[data["CKG"].notna()], col="CKG")

Large clusters  : 178
Coverage        : 70.0%


So we have a total of **51** clusters for GH and **179** clusters for KG.

# Sampling

## What is the representative sample size?

In [28]:
from bin.sampling import calculate_sample_size

In [29]:
print(f"GH Sample size: {calculate_sample_size(len(data[data["CGH"].notna()]))}")
print(f"KG Sample size: {calculate_sample_size(len(data[data["CKG"].notna()]))}")

GH Sample size: 383
KG Sample size: 384


## How much did we analyze?

In [31]:
GH = pd.read_csv("data/shome2023notebook/GH_sample-annot.csv")
GH[GH["EC"].isna()].shape

(382, 10)

In [36]:
GH[(GH["EC"].isna()) & (GH["Intent"].str.contains(r"^VAL"))].shape

(119, 10)

In [37]:
GH[(GH["EC"].isna()) & (~GH["Intent"].str.contains(r"^VAL"))].shape

(263, 10)

In [32]:
KG = pd.read_csv("data/shome2023notebook/KG_sample-annot.csv")
KG[KG["EC"].isna()].shape

(433, 10)

In [38]:
KG[(KG["EC"].isna()) & (KG["Intent"].str.contains(r"^VAL"))].shape

(4, 10)

In [39]:
KG[(KG["EC"].isna()) & (~KG["Intent"].str.contains(r"^VAL"))].shape

(429, 10)